# Open-Vocabulary 3D Segmentation
## coord.npy + color.npy → 텍스트로 물체 찾기

### 사용 방법
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 왼쪽 파일 패널에서 `coord.npy`, `color.npy` 업로드
3. 셀 순서대로 실행
4. 마지막 셀에서 찾고 싶은 물체 텍스트 입력

In [1]:
# ── 1. 설치 ──────────────────────────────────────────
%pip install open-clip-torch open3d scikit-learn -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── 2. 라이브러리 로드 ────────────────────────────────
import numpy as np
import torch
import open_clip
import open3d as o3d
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
디바이스: cuda


In [4]:
# ── 3. coord.npy / color.npy 로드 ────────────────────
coords = np.load('mosaic3d_input\\my_scan\\coord.npy').astype(np.float32)   # (N, 3) XYZ
colors = np.load('mosaic3d_input\\my_scan\\color.npy').astype(np.float32)   # (N, 3) RGB 0~255

# 0~1 범위로 정규화
if colors.max() > 1.0:
    colors = colors / 255.0

N = len(coords)
print(f'총 포인트 수: {N:,}개')
print(f'좌표 범위: X({coords[:,0].min():.2f}~{coords[:,0].max():.2f}), '
      f'Y({coords[:,1].min():.2f}~{coords[:,1].max():.2f}), '
      f'Z({coords[:,2].min():.2f}~{coords[:,2].max():.2f})')

총 포인트 수: 12,932,703개
좌표 범위: X(-3.85~4.42), Y(-4.72~2.84), Z(-2.32~1.18)


In [5]:
# ── 4. CLIP 모델 로드 ────────────────────────────────
print('CLIP 로드 중...')
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model = model.to(device).eval()
print('CLIP 로드 완료!')

CLIP 로드 중...


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

c:\Users\Gamzadole\anaconda3\envs\dreamLove\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Gamzadole\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Gamzadole\anaconda3\envs\dreamLove\Lib\site-packages\open_clip\factory.py:

CLIP 로드 완료!


In [6]:
# ── 5. 포인트 → 패치 이미지 변환 & CLIP 특징 추출 ────
# 각 포인트 주변의 RGB 색상으로 작은 패치 이미지를 만들어 CLIP에 넣기

from PIL import Image
import torchvision.transforms as T

PATCH_SIZE  = 16      # 패치 한 변 크기 (px)
CHUNK_SIZE  = 4096    # 한 번에 처리할 포인트 수
SUBSAMPLE   = 50000   # 전체에서 몇 개 샘플링할지 (메모리 절약)

# 서브샘플링 (1293만개 전부 처리하면 시간이 너무 걸림)
if N > SUBSAMPLE:
    sample_idx = np.random.choice(N, SUBSAMPLE, replace=False)
    coords_s   = coords[sample_idx]
    colors_s   = colors[sample_idx]
    print(f'서브샘플링: {N:,} → {SUBSAMPLE:,}개')
else:
    sample_idx = np.arange(N)
    coords_s   = coords
    colors_s   = colors

Ns = len(coords_s)

def make_patch_image(color_rgb):
    """RGB 색상 하나 → PATCH_SIZE x PATCH_SIZE 단색 이미지"""
    r, g, b = (color_rgb * 255).astype(np.uint8)
    img = Image.new('RGB', (PATCH_SIZE, PATCH_SIZE), (int(r), int(g), int(b)))
    return preprocess(img)

print(f'CLIP 특징 추출 중... ({Ns:,}개 포인트)')
all_features = []

for start in range(0, Ns, CHUNK_SIZE):
    end    = min(start + CHUNK_SIZE, Ns)
    batch  = colors_s[start:end]   # (chunk, 3)

    # 각 포인트 색상 → 패치 이미지 배치
    imgs = torch.stack([make_patch_image(c) for c in batch]).to(device)

    with torch.no_grad():
        feats = model.encode_image(imgs)          # (chunk, 512)
        feats = feats / feats.norm(dim=-1, keepdim=True)  # 정규화

    all_features.append(feats.cpu())

    if (start // CHUNK_SIZE) % 5 == 0:
        print(f'  {end:,}/{Ns:,} ({end/Ns*100:.1f}%)')

point_features = torch.cat(all_features, dim=0)  # (Ns, 512)
print(f'특징 추출 완료! shape: {point_features.shape}')

서브샘플링: 12,932,703 → 50,000개
CLIP 특징 추출 중... (50,000개 포인트)
  4,096/50,000 (8.2%)
  24,576/50,000 (49.2%)
  45,056/50,000 (90.1%)
특징 추출 완료! shape: torch.Size([50000, 512])


In [9]:
# ── 6. 텍스트 쿼리로 물체 찾기 ───────────────────────

def find_object(text_query, top_ratio=0.05, use_dbscan=True):
    """
    text_query  : 찾고 싶은 물체 (예: '장롱', '문', '소파')
    top_ratio   : 상위 몇 % 포인트를 해당 물체로 볼지
    use_dbscan  : DBSCAN으로 클러스터링해서 가장 큰 덩어리만 남길지
    """
    print(f'\n[검색] "{text_query}"')

    # 텍스트 → CLIP 특징
    prompts = [
        f'a photo of a {text_query}',
        f'a {text_query} in a room',
        f'{text_query}',
    ]
    tokens = tokenizer(prompts).to(device)
    with torch.no_grad():
        text_feats = model.encode_text(tokens)          # (3, 512)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
        text_feat  = text_feats.mean(dim=0, keepdim=True).cpu()  # (1, 512) 평균

    # 포인트별 유사도 계산 (코사인 유사도)
    sim = (point_features @ text_feat.T).squeeze(-1).numpy()  # (Ns,)

    # 상위 top_ratio% 포인트 선택
    
    threshold = np.percentile(sim, (1 - top_ratio) * 100)
    mask      = sim >= threshold
    matched_coords = coords_s[mask]
    matched_colors = colors_s[mask]

    print(f'  유사도 상위 {top_ratio*100:.1f}%: {mask.sum():,}개 포인트')

    # DBSCAN 클러스터링 → 가장 큰 덩어리만 남기기
    if use_dbscan and len(matched_coords) > 10:
        print('  DBSCAN 클러스터링 중...')
        db = DBSCAN(eps=0.3, min_samples=10).fit(matched_coords)
        labels = db.labels_

        if labels.max() >= 0:  # 클러스터가 하나라도 있으면
            # 가장 큰 클러스터 선택
            largest = np.bincount(labels[labels >= 0]).argmax()
            cluster_mask   = labels == largest
            matched_coords = matched_coords[cluster_mask]
            matched_colors = matched_colors[cluster_mask]
            print(f'  가장 큰 클러스터: {cluster_mask.sum():,}개 포인트')

    # 중심 좌표 계산
    centroid = matched_coords.mean(axis=0)
    print(f'  중심 좌표 (X, Y, Z): ({centroid[0]:.3f}, {centroid[1]:.3f}, {centroid[2]:.3f})')

    return matched_coords, matched_colors, centroid, sim


def visualize_3d_interactive(query, matched_coords, matched_colors, centroid, sample=30000):
    """
    Plotly로 인터랙티브 3D 뷰어 (Colab에서 마우스로 회전/줌 가능)
    """
    import plotly.graph_objects as go
 
    # 전체 포인트 서브샘플링
    idx = np.random.choice(len(coords_s), min(sample, len(coords_s)), replace=False)
    bg_coords = coords_s[idx]
    bg_colors = colors_s[idx]
 
    # RGB → plotly 색상 문자열 변환
    def to_rgb_str(color_array):
        return [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"
                for r, g, b in color_array]
 
    fig = go.Figure()
 
    # 1) 배경 포인트 클라우드 (원본 RGB)
    fig.add_trace(go.Scatter3d(
        x=bg_coords[:, 0],
        y=bg_coords[:, 1],
        z=bg_coords[:, 2],
        mode='markers',
        marker=dict(
            size=1,
            color=to_rgb_str(bg_colors),
            opacity=0.4,
        ),
        name='전체 포인트',
        hoverinfo='skip',
    ))
 
    # 2) 감지된 물체 포인트 (빨간색)
    fig.add_trace(go.Scatter3d(
        x=matched_coords[:, 0],
        y=matched_coords[:, 1],
        z=matched_coords[:, 2],
        mode='markers',
        marker=dict(
            size=3,
            color='red',
            opacity=0.9,
        ),
        name=f'감지: {query}',
    ))
 
    # 3) 중심 좌표 (노란 별)
    fig.add_trace(go.Scatter3d(
        x=[centroid[0]],
        y=[centroid[1]],
        z=[centroid[2]],
        mode='markers+text',
        marker=dict(
            size=10,
            color='yellow',
            symbol='diamond',
            line=dict(color='black', width=1),
        ),
        text=[f'중심<br>({centroid[0]:.2f}, {centroid[1]:.2f}, {centroid[2]:.2f})'],
        textposition='top center',
        name='중심 좌표',
    ))
 
    fig.update_layout(
        title=dict(
            text=f'Open-Vocabulary 3D Search: "{query}"',
            font=dict(size=18),
        ),
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            bgcolor='rgb(10, 10, 10)',
            xaxis=dict(showgrid=False, zeroline=False),
            yaxis=dict(showgrid=False, zeroline=False),
            zaxis=dict(showgrid=False, zeroline=False),
        ),
        paper_bgcolor='rgb(20, 20, 20)',
        font=dict(color='white'),
        legend=dict(
            font=dict(size=12),
            bgcolor='rgba(0,0,0,0.5)',
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=700,
    )
 
    fig.show()
    print(f'\n중심 좌표: X={centroid[0]:.3f}, Y={centroid[1]:.3f}, Z={centroid[2]:.3f}')
    print(f'드론 목표 좌표: {centroid.tolist()}')

In [10]:
# ── 실행 ──────────────────────────────────────────────
QUERY = 'chair'
 
matched_coords, matched_colors, centroid, sim = find_object(
    text_query=QUERY,
    top_ratio=0.10,
    use_dbscan=True
)
 
visualize_3d_interactive(QUERY, matched_coords, matched_colors, centroid, sample=30000)
 
print(f'\n===== 결과 요약 =====')
print(f'  쿼리:        "{QUERY}"')
print(f'  감지 포인트: {len(matched_coords):,}개')
print(f'  중심 좌표:   X={centroid[0]:.3f}, Y={centroid[1]:.3f}, Z={centroid[2]:.3f}')
print(f'  → 드론 목표 좌표: {centroid.tolist()}')


[검색] "chair"
  유사도 상위 10.0%: 5,000개 포인트
  DBSCAN 클러스터링 중...
  가장 큰 클러스터: 4,159개 포인트
  중심 좌표 (X, Y, Z): (-0.239, 0.629, -1.217)



중심 좌표: X=-0.239, Y=0.629, Z=-1.217
드론 목표 좌표: [-0.23917537927627563, 0.629226803779602, -1.2174322605133057]

===== 결과 요약 =====
  쿼리:        "chair"
  감지 포인트: 4,159개
  중심 좌표:   X=-0.239, Y=0.629, Z=-1.217
  → 드론 목표 좌표: [-0.23917537927627563, 0.629226803779602, -1.2174322605133057]
